In [57]:
from dotenv import load_dotenv
from sqlalchemy import create_engine, text, bindparam
from datetime import date, datetime, timedelta
from pathlib import Path
import pandas as pd
import os
import urllib3

# Load environment variables from .env file
load_dotenv()

strPresto = ('presto://{username}:{password}@{ipaddress}:{port}/{dbname}/{schema}'
             .format(username=os.getenv('HIVE_SVC_USER'),
                     password=os.getenv('HIVE_SVC_PASS'),
                     ipaddress=os.getenv('HIVE_SVC_ADDRESS'),
                     port=os.getenv('HIVE_SVC_PORT'),
                     dbname=os.getenv('HIVE_SVC_DBNAME'),
                     schema=os.getenv('HIVE_SVC_SCHEMA')))
 
presto_engine = create_engine(strPresto, connect_args={"protocol": "https", "requests_kwargs": {"verify": False}})

# disable certificate warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [58]:
# ------------------------------------------------------------------
# Start date and end dates for dateselectors in queries
# ------------------------------------------------------------------

#start_date = '2025-12-13' 
#end_date = '2026-02-06' 

today = date.today()
diff_to_friday = (4 - today.weekday()) % 7  # Mon=0 ... Fri=4

# Go back 7 weeks since we are counting the current week
eight_weeks_ago = today - timedelta(weeks=7) 

# Find the Saturday of that week (Mon=0 ... Sun=6, Sat=5)
days_since_saturday = (eight_weeks_ago.weekday() - 5) % 7

start_saturday = eight_weeks_ago - timedelta(days=days_since_saturday)
end_friday = today + timedelta(days=diff_to_friday)

#format date as string for Presto query
start_date = start_saturday.strftime('%Y-%m-%d')
end_date = end_friday.strftime('%Y-%m-%d')


In [59]:
start_date

'2025-12-27'

In [60]:
# ------------------------------------------------------------------
# ICP Client list as stored in hive.care.expert_performance_metrics.metric (lowercase)
# ------------------------------------------------------------------

icp_client_list = [
    "pss-verizon",
    "pss-at&t",
    "mob-verizon",
    "mob-at&t"
]

In [61]:
# ------------------------------------------------------------------
# Metric configuration
# metric_list are the metric names as stored in hive.care.expert_performance_metrics.metric (lowercase)
# metric_name_map is optional for pretty display names
# ------------------------------------------------------------------

metric_list = [
    "erp",
    "nsp100",
    "cancellation rate",
    "transfers",
    "resolution rate",
    "crt"
]

metric_name_map = {
    "erp": "ERP",
    "nsp100": "NSP 100",
    "cancellation rate": "Cancel Rate",
    "transfers": "Transfers",
    "resolution rate": "Resolution",
    "crt": "CRT",
}

In [62]:
# ------------------------------------------------------------------
# Load SQL template for metric query
# ------------------------------------------------------------------

sql_path = "SQL/epm_expert_data_week.sql"  # <- make sure this path is correct

with open(sql_path, "r") as f:
    METRIC_SQL_TEMPLATE = f.read()

print("Loaded SQL template:")
print(METRIC_SQL_TEMPLATE[:500], "...")

Loaded SQL template:
SELECT 
CAST(week_stop_date AS DATE) AS week_ending,
expert_id,
'all' AS call_type,
metric,
SUM(numerator) AS num,
SUM(denominator) AS den,
ROUND(
COALESCE(
CAST(SUM(numerator) AS DOUBLE) /
NULLIF(CAST(SUM(denominator) AS DOUBLE), 0.0),
0.000
),
3
) AS calc
FROM 
hive.care.expert_performance_metrics a
LEFT OUTER JOIN 
hive.care.l4_asurion_umt_ppx_pay_calendar d ON a."date" = CAST(d.event_date AS DATE)
WHERE 
LOWER(metric) IN :metric_list
AND LOWER(icp_client) IN :icp_client_list
AND a."date" bet ...


In [63]:
### Compile SQL With Literal Binds (Code)

#This uses your proven pattern: bind params + `literal_binds=True`.

#python
# ------------------------------------------------------------------
# Build literal SQL for Presto using SQLAlchemy binds
# This allows us to use expanding=True for metric_list and still
# send flattened literal SQL to Presto.
# ------------------------------------------------------------------

def compile_presto_sql(
    sql_template: str,
    engine,
    start_date,
    end_date,
    icp_client_list,
    metric_list,
):
    """
    Creates literal SQL for Presto by binding parameters and compiling
    with literal_binds=True.
    """
    
    stmt = text(sql_template).bindparams(
        bindparam("start_date", value=start_date),
        bindparam("end_date", value=end_date),
        bindparam("icp_client_list", value=list(icp_client_list), expanding=True),
        bindparam("metric_list", value=list(metric_list), expanding=True),
    )

    compiled = stmt.compile(
        engine,
        compile_kwargs={"literal_binds": True}
    )

    return str(compiled)


In [64]:
# ------------------------------------------------------------------
# Query Presto for metrics, client groups, and date range, returning the
# aggregated metrics DataFrame.
# ------------------------------------------------------------------

def query_metrics_presto_group(
    start_date,
    end_date,
    icp_client_list,
    metric_list,
):
    """
    Execute the Presto query for a list of experts over a date range.

    Returns DataFrame with:
      [expert_id, metric, icp_client, site, num, den, calc]
    """

    sql = compile_presto_sql(
        sql_template=METRIC_SQL_TEMPLATE,
        engine=presto_engine,
        start_date=start_date,
        end_date=end_date,
        icp_client_list=icp_client_list,
        metric_list=metric_list,
    )

    # Uncomment to debug generated SQL:
    # print(sql)

    with presto_engine.connect() as conn:
        df = pd.read_sql(sql, conn)

    # Normalize types for downstream joins
    if not df.empty:
        df["metric"] = df["metric"].str.lower()

    return df


In [65]:
df = query_metrics_presto_group(
        start_date,
        end_date,
        icp_client_list,
        metric_list,
    )

In [66]:
df

,week_ending,expert_id,call_type,metric,num,den,calc
0,2026-01-02,699383,all,crt,67052.0,43.0,1559.349
1,2026-01-02,685174,all,nsp100,5.0,41.0,0.122
2,2026-01-02,679826,all,resolution rate,23.0,24.0,0.958
3,2026-01-02,699436,all,crt,78107.0,45.0,1735.711
4,2026-01-02,698072,all,resolution rate,16.0,17.0,0.941
...,...,...,...,...,...,...,...
122701,2026-01-16,483527,all,erp,100.0,1.0,100.000
122702,2026-01-02,699432,all,nsp100,NaN,2.0,0.000
122703,2026-01-02,692846,all,cancellation rate,1.0,NaN,0.000
122704,2026-01-30,702704,all,cancellation rate,NaN,2.0,0.000


In [67]:
# Save to CSV in the same directory

from pathlib import Path

file = Path("../data/raw/weekly/2026-02-16/agent_metrics.csv")   # replace with your filename

if file.exists():
    file.unlink()
    df.to_csv("../data/raw/weekly/2026-02-16/agent_metrics.csv", index=False)
else:
    df.to_csv("../data/raw/weekly/2026-02-16/agent_metrics.csv", index=False)
